# Análise da equipe: comparação P vs. N

Este notebook registra a análise descrita pela Pessoa 1 para o Marco 3. A hipótese confirmatória compara a proporção de acertos por participante em P e N; a unidade é o participante, com 65 pares. A análise principal usa `results-03-final.csv` sem nova limpeza, mantém oito durações abaixo de 10 s e calcula as médias com respostas válidas disponíveis. O teste principal é t pareado bilateral (α = 0,05); Wilcoxon pareado é a verificação de robustez. As análises por sequência, com filtro de duração e por padrão são exploratórias.

O relatório apresenta dois efeitos bisseriais para Wilcoxon com convenções diferentes para os empates: r = 0,56 na amostra completa (n = 65), usando a fórmula do slide da disciplina, e r = 0,71 excluindo os 26 empates (n = 39). O código abaixo calcula explicitamente a estimativa entre os 39 participantes não empatados. A estimativa de 0,56 e seu IC bootstrap são os valores produzidos pelo script da Pessoa 1 com a fórmula do slide.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
from statsmodels.stats.multitest import multipletests

DATA_PATH = Path('dados/brutos/results-03-final.csv')
ALPHA = 0.05
BOOTSTRAP_REPS = 10_000
BOOTSTRAP_SEED = 2026
MIN_TASK_SECONDS = 10
MAX_TASK_SECONDS = 400


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Linhas no arquivo final: {len(df)}')
print(f'Sequências: {df["randnumber"].value_counts().sort_index().to_dict()}')
print(f'ID 54 presente: {(df["id"] == 54).any()}')
df.head()


## Preparação da medida confirmatória

Para cada participante, calculamos a média das respostas binárias disponíveis nas seis tarefas P e nas seis tarefas N. Os quatro campos ausentes permanecem ausentes (IDs 63 e 65 em WL1P, ID 89 em ME2P e ID 80 em AG1N); assim, cada condição de cada participante ainda tem uma média calculável. Os oito tempos abaixo de 10 s permanecem na análise principal, conforme o relatório da Pessoa 1.


In [ ]:
patterns = ['ER', 'PT', 'WL', 'ME', 'AG', 'RB']
scenarios = [1, 2]
p_cols = [f'{pattern}{scenario}P' for pattern in patterns for scenario in scenarios]
n_cols = [f'{pattern}{scenario}N' for pattern in patterns for scenario in scenarios]
all_score_cols = p_cols + n_cols
missing_columns = sorted(set(all_score_cols) - set(df.columns))
if missing_columns:
    raise KeyError(f'Colunas de corretude ausentes: {missing_columns}')

responses = df[all_score_cols].apply(pd.to_numeric, errors='coerce')
summary = pd.DataFrame({
    'id': df['id'],
    'sequence': df['randnumber'],
    'n_valid_P': responses[p_cols].notna().sum(axis=1),
    'n_valid_N': responses[n_cols].notna().sum(axis=1),
    'P': responses[p_cols].mean(axis=1),
    'N': responses[n_cols].mean(axis=1),
})
summary['difference_P_minus_N'] = summary['P'] - summary['N']
paired = summary.dropna(subset=['P', 'N']).copy()
print('Pares completos:', len(paired))
print('Tarefas válidas P/N:', int(responses[p_cols].notna().sum().sum()), '/', int(responses[n_cols].notna().sum().sum()))
print('Contagem n_valid_P:', summary['n_valid_P'].value_counts().sort_index().to_dict())
print('Contagem n_valid_N:', summary['n_valid_N'].value_counts().sort_index().to_dict())


In [ ]:
descriptive = paired[['P', 'N', 'difference_P_minus_N']].agg(['count', 'mean', 'std', 'median', 'min', 'max']).T
descriptive = descriptive.rename(columns={'count':'n', 'mean':'mean', 'std':'SD', 'median':'median', 'min':'min', 'max':'max'})
descriptive['Q1'] = paired[['P', 'N', 'difference_P_minus_N']].quantile(0.25).values
descriptive['Q3'] = paired[['P', 'N', 'difference_P_minus_N']].quantile(0.75).values
descriptive * 100  # proporções/diferenças em pontos percentuais para relatório


In [ ]:
# Gráfico pareado: cada linha liga o mesmo participante nas condições N e P.
fig, ax = plt.subplots(figsize=(7, 5))
for _, row in paired.iterrows():
    color = '#27864b' if row['P'] > row['N'] else ('#888888' if row['P'] == row['N'] else '#c43c39')
    ax.plot([0, 1], [row['N'] * 100, row['P'] * 100], color=color, alpha=0.45, linewidth=0.8)
means = paired[['N', 'P']].mean() * 100
sem = paired[['N', 'P']].sem() * 100
crit = stats.t.ppf(1 - ALPHA / 2, len(paired) - 1)
ax.errorbar([0, 1], means, yerr=crit * sem, fmt='D', capsize=5, color='black', zorder=3)
ax.set_xticks([0, 1], ['Sem padrão (N)', 'Com padrão (P)'])
ax.set_ylabel('Proporção de respostas corretas (%)')
ax.set_ylim(0, 105)
ax.set_title('Corretude por participante e condição (n = 65 pares)')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()


## Teste confirmatório e verificação de robustez

O teste t pareado é o principal porque a hipótese é sobre a diferença média por participante. A distribuição das diferenças é discreta e tem muitos zeros; Shapiro–Wilk é reportado como diagnóstico, não como seletor automático do teste. Wilcoxon foi executado como análise de robustez após a rejeição de normalidade.


In [ ]:
d = paired['difference_P_minus_N'].to_numpy(dtype=float)
n = len(d)
if n != 65:
    print(f'ATENÇÃO: n pareado atual ({n}) difere de 65 no relatório da Pessoa 1.')

shapiro = stats.shapiro(d)
test_t = stats.ttest_rel(paired['P'], paired['N'], alternative='two-sided')
mean_d = float(np.mean(d))
sd_d = float(np.std(d, ddof=1))
se_d = sd_d / np.sqrt(n)
t_critical = stats.t.ppf(1 - ALPHA / 2, n - 1)
ci_mean_d = (mean_d - t_critical * se_d, mean_d + t_critical * se_d)
d_z = mean_d / sd_d

# IC 95% para dz por inversão da distribuição t não central.
def ncp_root(target):
    return brentq(lambda ncp: stats.nct.cdf(test_t.statistic, n - 1, ncp) - target, -1000, 1000)
ncp_low = ncp_root(1 - ALPHA / 2)
ncp_high = ncp_root(ALPHA / 2)
dz_ci = (ncp_low / np.sqrt(n), ncp_high / np.sqrt(n))

wilcoxon = stats.wilcoxon(paired['P'], paired['N'], alternative='two-sided', zero_method='wilcox', method='approx')
n_nonzero = int(np.count_nonzero(d))
rank_biserial_nonzero = np.nan
if n_nonzero > 0:
    rank_biserial_nonzero = 1 - 2 * float(wilcoxon.statistic) / (n_nonzero * (n_nonzero + 1) / 2)

# IC bootstrap pareado (reamostragem de participantes/diferenças; semente fixa).
rng = np.random.default_rng(BOOTSTRAP_SEED)
boot_idx = rng.integers(0, n, size=(BOOTSTRAP_REPS, n))
boot_d = d[boot_idx]
boot_mean = boot_d.mean(axis=1)
boot_ci_mean = tuple(np.quantile(boot_mean, [ALPHA / 2, 1 - ALPHA / 2]))
def rb_from_diffs(values):
    values = np.asarray(values, dtype=float)
    values = values[values != 0]
    if len(values) == 0:
        return np.nan
    ranks = stats.rankdata(np.abs(values))
    r_plus = ranks[values > 0].sum()
    r_minus = ranks[values < 0].sum()
    return float((r_plus - r_minus) / (r_plus + r_minus))
boot_rb_nonzero = np.array([rb_from_diffs(row) for row in boot_d])
boot_ci_rb_nonzero = tuple(np.nanquantile(boot_rb_nonzero, [ALPHA / 2, 1 - ALPHA / 2]))

print(f'Shapiro–Wilk: W={shapiro.statistic:.3f}, p={shapiro.pvalue:.3g}')
print(f't pareado: t({n-1})={test_t.statistic:.2f}, p={test_t.pvalue:.3g}')
print(f'Diferença média P−N: {mean_d*100:.1f} p.p.; IC 95% [{ci_mean_d[0]*100:.1f}, {ci_mean_d[1]*100:.1f}]')
print(f'Cohen dz={d_z:.2f}; IC 95% não central [{dz_ci[0]:.2f}, {dz_ci[1]:.2f}]')
print(f'Diferenças nulas={int(np.sum(d == 0))}; diferenças não nulas={n_nonzero}')
print(f'Wilcoxon T={wilcoxon.statistic:.0f}, p={wilcoxon.pvalue:.3g}, n efetivo={n_nonzero}')
print(f'r bisserial entre os {n_nonzero} participantes sem empate={rank_biserial_nonzero:.3f}; IC bootstrap sem empates={boot_ci_rb_nonzero}')
print(f'IC bootstrap da diferença média P−N: [{boot_ci_mean[0]*100:.1f}, {boot_ci_mean[1]*100:.1f}] p.p.')
print('O efeito na amostra completa (n=65), calculado pela fórmula do slide da disciplina, é reportado pela Pessoa 1 como r=0,56 com IC bootstrap [0,34; 0,75].')


## Análises exploratórias descritas pela Pessoa 1

Estas análises são verificações de sensibilidade e não alteram a hipótese confirmatória.


In [ ]:
# Diferenças médias por sequência e estimador com peso igual para as duas sequências.
seq_table = paired.groupby('sequence').agg(n=('id', 'size'), mean_P=('P', 'mean'), mean_N=('N', 'mean'))
seq_table['mean_difference'] = seq_table['mean_P'] - seq_table['mean_N']
seq_table


In [ ]:
# Comparação exploratória das diferenças entre sequências (Welch).
seq_values = [g['difference_P_minus_N'].to_numpy() for _, g in paired.groupby('sequence')]
if len(seq_values) == 2:
    welch_seq = stats.ttest_ind(seq_values[0], seq_values[1], equal_var=False)
    grouped = paired.groupby('sequence')['difference_P_minus_N']
    seq_means = grouped.mean()
    seq_vars = grouped.var(ddof=1)
    seq_ns = grouped.count()
    balanced_estimate = float(seq_means.mean())
    balanced_se2 = float((seq_vars / seq_ns).sum() / 4)
    balanced_se = np.sqrt(balanced_se2)
    balanced_df = balanced_se2**2 / float((((seq_vars / seq_ns)**2) / (seq_ns - 1)).sum() / 16)
    balanced_crit = stats.t.ppf(1 - ALPHA / 2, balanced_df)
    balanced_ci = (balanced_estimate - balanced_crit * balanced_se, balanced_estimate + balanced_crit * balanced_se)
    balanced_t = balanced_estimate / balanced_se
    balanced_p = 2 * stats.t.sf(abs(balanced_t), balanced_df)
    print(f'Welch entre sequências: t={welch_seq.statistic:.2f}, p={welch_seq.pvalue:.3g}')
    print(f'Diferença média balanceada entre sequências: {balanced_estimate*100:.1f} p.p.; IC 95% [{balanced_ci[0]*100:.1f}, {balanced_ci[1]*100:.1f}]; t={balanced_t:.2f}, gl={balanced_df:.1f}, p={balanced_p:.3g}')


In [ ]:
# Sensibilidade: mascarar tarefa se duração <10 s ou >400 s e recalcular média P/N.
time_cols = [f'{col}Time' for col in all_score_cols]
missing_times = sorted(set(time_cols) - set(df.columns))
if missing_times:
    print('Não foi possível executar o filtro de duração; faltam colunas:', missing_times)
else:
    durations = df[time_cols].apply(pd.to_numeric, errors='coerce')
    valid_duration = (durations >= MIN_TASK_SECONDS) & (durations <= MAX_TASK_SECONDS)
    duration_by_score = {t: t + 'Time' for t in all_score_cols}
    filtered = responses.copy()
    for score_col in all_score_cols:
        filtered[score_col] = filtered[score_col].where(valid_duration[duration_by_score[score_col]])
    filtered_P = filtered[p_cols].mean(axis=1)
    filtered_N = filtered[n_cols].mean(axis=1)
    filtered_pairs = pd.DataFrame({'P': filtered_P, 'N': filtered_N}).dropna()
    filtered_t = stats.ttest_rel(filtered_pairs['P'], filtered_pairs['N'])
    filtered_d = filtered_pairs['P'] - filtered_pairs['N']
    print('Pares após filtro:', len(filtered_pairs))
    removed = sum(int((responses[col].notna() & ~valid_duration[f'{col}Time']).sum()) for col in all_score_cols)
    filtered_w = stats.wilcoxon(filtered_pairs['P'], filtered_pairs['N'], alternative='two-sided', zero_method='wilcox', method='approx')
    filtered_dz = filtered_d.mean() / filtered_d.std(ddof=1)
    print('Respostas observadas removidas por duração:', removed)
    print(f'Diferença média: {filtered_d.mean()*100:.1f} p.p.; t={filtered_t.statistic:.2f}, p={filtered_t.pvalue:.3g}, dz={filtered_dz:.2f}')
    print(f'Wilcoxon de sensibilidade: p={filtered_w.pvalue:.3g}')


In [ ]:
# Comparação exploratória por padrão: McNemar exato (P vs N, pareado) e correção de Holm.
pattern_rows = []
for pattern in patterns:
    p_task_cols = [f'{pattern}{scenario}P' for scenario in scenarios]
    n_task_cols = [f'{pattern}{scenario}N' for scenario in scenarios]
    p_score = responses[p_task_cols].sum(axis=1, min_count=1)
    n_score = responses[n_task_cols].sum(axis=1, min_count=1)
    valid = p_score.notna() & n_score.notna()
    # Em cada sequência, cada participante tem uma tarefa P e uma N por padrão.
    p_binary = (p_score[valid] == 1).astype(int)
    n_binary = (n_score[valid] == 1).astype(int)
    both = pd.crosstab(p_binary, n_binary).reindex(index=[0,1], columns=[0,1], fill_value=0)
    p_only = int(both.loc[1, 0])
    n_only = int(both.loc[0, 1])
    discordant = p_only + n_only
    pvalue = stats.binomtest(min(p_only, n_only), n=discordant, p=0.5, alternative='two-sided').pvalue if discordant else 1.0
    pattern_rows.append({'pattern': pattern, 'n': int(valid.sum()), 'mean_P': p_score[valid].mean(), 'mean_N': n_score[valid].mean(), 'p_only': p_only, 'n_only': n_only, 'p_exact': pvalue})
pattern_results = pd.DataFrame(pattern_rows)
pattern_results['p_holm'] = multipletests(pattern_results['p_exact'], alpha=ALPHA, method='holm')[1]
pattern_results


## Registro para transferir ao artigo

- No Método, descrever a variável confirmatória, o arquivo final, 65 pares, tratamento de ausências, teste principal e verificação de robustez.
- Na seção de Resultados, usar os números da execução final validada pela Pessoa 1.
- Rotular análises por sequência, duração e padrão como exploratórias.
- Na tabela do Wilcoxon, distinguir r = 0,56 (amostra completa, n = 65, fórmula do slide; IC bootstrap [0,34; 0,75]) de r = 0,71 (sem os 26 empates, n = 39).
